# 03 · Preprocessing — frozen splits and modelling hand-off

**Project:** Skin Lesion Triage · BIU DS23 Capstone  
**Owner:** Data hat · **Charter sections:** 3 (Data) and 6 (Risks)

This notebook is the single source of truth for modelling manifests.

> **Inputs this notebook requires.** From notebook 01: `data/raw/isic_clinical_metadata.csv`
> (with the `label_family` column) and `data/processed/isic_duplicate_exclusions.csv`. From
> notebook 02: `data/processed/pad_split.csv` and `data/raw/pad_metadata_raw.csv`. The PAD
> photographs are fetched by whichever of notebooks 01 and 03 runs first. Run order: 01 → 02 → 03.

- The PAD patient-level split was created and frozen in notebook 02 and is only read here.
- The ISIC train/validation split is created here once, saved with the manifest, and loaded on every later run.
- Downstream modelling notebooks must reuse the manifests written here and must not create new splits.

## Outputs

| File | What it is |
|---|---|
| `data/processed/isic_image_manifest.csv` + `data/raw/images/images_isic_512.tar` | the frozen ISIC pool and split, 6,404 images. The image model may consume it directly or rebuild an equivalent index from the same sources and the same rules — the CV branch does the latter and reproduces every count |
| `data/processed/pad_image_manifest.csv` + `data/raw/images/pad_original/` | the frozen PAD patient split, 2,298 images. Fine-tuning uses train patients, validation is for model selection, test is opened once |
| `data/processed/pad_symptom_features.csv` | lesion-level structured features for the symptom model |
| `data/processed/aggregation_policy.json` | image → lesion → patient scoring: max, then max |

## Decisions applied here

1. ISIC exclusions decided in notebook 01 are applied here: 124 Indeterminate, 9 collision lesions and 2 duplicate photographs. The label follows the frozen MEL + BCC + SCC policy via `label_family`.
2. In PAD, malignant means MEL + BCC + SCC; ACK remains non-malignant for the project target.
3. Splitting is patient-level wherever a patient identifier exists, otherwise lesion-level.
4. Symptom modelling uses one row per patient-lesion rather than one row per photograph.
5. Predictive feature lists are explicit; outcome-derived and administratively leaky fields are excluded.
6. PAD image files come from a public mirror; their content is verified against PAD's copy inside ISIC (Cell 5e).

In [ ]:
# Cell 1 · Configuration (identical in every notebook)
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = "/content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers"
else:
    PROJECT = os.path.abspath("..")

RAW       = os.path.join(PROJECT, "data", "raw")
PROCESSED = os.path.join(PROJECT, "data", "processed")
FIGURES   = os.path.join(PROJECT, "reports", "figures")

# Where the image archive is stored. Kept as its own setting so the heavy files can later move to a
# shared folder without touching anything else. Colab addresses Drive by path, never by the id in a
# share link, so any override below must be a path under /content/drive/MyDrive/.
IMAGES_DIR = os.path.join(RAW, "images")

for d in (RAW, PROCESSED, FIGURES, IMAGES_DIR):
    os.makedirs(d, exist_ok=True)

SEED = 42              # every split in this project uses this seed
IMG_SIZE = 512         # long side, in pixels, for the training copy of the images

print("project  :", PROJECT)
print("raw      :", RAW)
print("processed:", PROCESSED)
print("images   :", IMAGES_DIR)


Mounted at /content/drive
project  : /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers
raw      : /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/raw
processed: /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/processed
images   : /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/raw/images


In [ ]:
# Cell 2 · Load what notebooks 01 and 02 produced
import pandas as pd, numpy as np

ISIC_META = os.path.join(RAW, "isic_clinical_metadata.csv")
PAD_META  = os.path.join(RAW, "pad_metadata_raw.csv")

for path, producer in [(ISIC_META, "notebook 01, Cell 9"), (PAD_META, "notebook 02, Cell 2")]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found - it is written by {producer}.")

isic = pd.read_csv(ISIC_META)
pad  = pd.read_csv(PAD_META).replace({"UNK": np.nan, "": np.nan})

# label_family was added to notebook 01 after the first version of this notebook, and three cells
# below depend on it. An older CSV would fail much later, in a place that does not explain itself.
assert "label_family" in isic.columns, \
    "isic_clinical_metadata.csv has no label_family column - re-run notebook 01 Cells 5b and 9"

print(f"ISIC metadata : {len(isic):,} rows  (all clinical close-ups, including the excluded ones)")
print(f"PAD metadata  : {len(pad):,} rows")
print(f"\nISIC training pool: {int(isic['in_training_pool'].sum()):,}")
print(isic["exclusion_reason"].value_counts(dropna=False).to_string())


ISIC metadata : 9,316 rows  (all clinical close-ups, including the excluded ones)
PAD metadata  : 2,298 rows

ISIC training pool: 6,539
exclusion_reason
NaN                     6539
pad_test_set            2298
unlabelled_benchmark     479


## 1 · ISIC — the training pool

The exclusions decided in notebook 01 are applied in one place: Indeterminate rows, collision lesions whose malignant component is unidentified, and duplicate photographs. Every drop is counted and printed, because "how many rows did you drop and why" is a question the charter answers with numbers.

> ### Why the grouping key is patient first, lesion second
> Notebook 01 measured both. Images of the same **lesion** are rare: only 0.2% of lesions have more than
> one image, so a naive split leaks just 1.0% of test images — lesion-level grouping is almost a no-op here.
>
> The concentration is at **patient** level: `patient_id` is filled for 19.1% of the pool, and those
> 1,247 images come from **79 patients** (1,241 from Memorial Sloan Kettering, 6 from Hospital Italiano).
> Grouping by lesion would scatter one patient's photos across train and validation.
>
> So the key is `patient_id` where it exists, `lesion_id` where it does not, and the image itself as a last
> resort. This is also exactly what the charter's risk table promises.

In [ ]:
# Cell 3 · Build the labelled ISIC modelling pool
# Exclusions decided in notebook 01, applied here in one place:
#   124 Indeterminate, 9 collision lesions (label_family OTHER), 2 duplicate photographs.
# The label follows the frozen policy: MEL + BCC + SCC = 1, everything else = 0.

pool = isic[isic["in_training_pool"] == True].copy()
n_before = len(pool)

dup_path = os.path.join(PROCESSED, "isic_duplicate_exclusions.csv")
if not os.path.exists(dup_path):
    raise FileNotFoundError(
        f"{dup_path} not found. It is written by notebook 01, Cell 16d, after the duplicate "
        f"verdicts are recorded. Run notebook 01 to the end before building the modelling pool.")
dup_ids = set(pd.read_csv(dup_path)["isic_id"])

drop_indet = pool["label_family"].eq("INDETERMINATE")
drop_other = pool["label_family"].eq("OTHER")
drop_dup   = pool["isic_id"].isin(dup_ids)

print(f"training pool (candidates)  : {n_before:,}")
print(f"  dropped, Indeterminate     : {int(drop_indet.sum()):,}")
print(f"  dropped, collision (OTHER) : {int(drop_other.sum()):,}")
print(f"  dropped, duplicate photo   : {int(drop_dup.sum()):,}")

pool = pool[~(drop_indet | drop_other | drop_dup)].copy()
assert pool["label_family"].isin(["MEL", "BCC", "SCC", "BENIGN"]).all(), \
    "a row outside the frozen label policy survived"

pool["label"] = pool["label_family"].isin(["MEL", "BCC", "SCC"]).astype(int)
pool["is_melanoma"] = pool["is_melanoma"].fillna(0).astype(int)

assert len(pool) == 6404, f"expected 6,404 images after exclusions, found {len(pool):,}"
assert int(pool["label"].sum()) == 3687, f"expected 3,687 malignant, found {int(pool['label'].sum()):,}"
assert int(pool["is_melanoma"].sum()) == 470, f"expected 470 melanomas, found {int(pool['is_melanoma'].sum())}"

print(f"\nlabelled modelling pool     : {len(pool):,}")
print(f"  malignant                  : {int(pool['label'].sum()):,} ({100 * pool['label'].mean():.1f}%)")
print(f"  melanomas                  : {int(pool['is_melanoma'].sum()):,}")
print("\nlabel families:")
print(pool["label_family"].value_counts().to_string())

# Group at the strictest available level; prefixes stop ids from different sources colliding
source_key  = pool["attribution"].fillna("unknown_source").astype(str)
patient_key = pool["patient_id"].astype(str)
lesion_key  = pool["lesion_id"].astype(str)

pool["group_id"] = np.where(
    pool["patient_id"].notna(),
    source_key + "::P::" + patient_key,
    np.where(
        pool["lesion_id"].notna(),
        source_key + "::L::" + lesion_key,
        source_key + "::I::" + pool["isic_id"].astype(str),
    ),
)

sizes = pool.groupby("group_id").size()
print(f"\ndistinct groups: {len(sizes):,} for {len(pool):,} images")
print(f"largest group: {sizes.max()} images")
print(f"groups with more than one image: {int((sizes > 1).sum()):,}")

training pool (candidates)  : 6,539
  dropped, Indeterminate     : 124
  dropped, collision (OTHER) : 9
  dropped, duplicate photo   : 2

labelled modelling pool     : 6,404
  malignant                  : 3,687 (57.6%)
  melanomas                  : 470

label families:
label_family
BENIGN    2717
BCC       2549
SCC        668
MEL        470

distinct groups: 5,237 for 6,404 images
largest group: 24 images
groups with more than one image: 72


In [ ]:
# Cell 4 · Create the canonical ISIC train / validation split ONCE, then reuse it
# Stratified by contributor AND label, not by label alone: notebook 01 showed that the contributor
# nearly determines the diagnosis (MSK 99.4% benign, MILK 69.4% malignant; contributor-only AUC 0.729).
from sklearn.model_selection import StratifiedGroupKFold

ISIC_MANIFEST = os.path.join(PROCESSED, "isic_image_manifest.csv")
loaded = False

if os.path.exists(ISIC_MANIFEST):
    old = pd.read_csv(ISIC_MANIFEST, usecols=["isic_id", "split"])
    if set(old["isic_id"]) == set(pool["isic_id"]):
        pool["split"] = pool["isic_id"].map(old.set_index("isic_id")["split"])
        loaded = True
        print("ISIC split LOADED from:", ISIC_MANIFEST)
    else:
        print(f">>> The saved manifest has {len(old):,} images; the pool now has {len(pool):,}.")
        print(">>> The split is re-created ONCE and frozen from here on.")
        print(">>> Any image model trained on the old manifest must be retrained on the new one.")

if not loaded:
    pool["stratum"] = pool["attribution"].astype(str).str[:20] + "_" + pool["label"].astype(str)
    sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)   # 1/6 -> ~17% validation
    train_idx, val_idx = next(sgkf.split(pool, pool["stratum"], groups=pool["group_id"]))
    pool["split"] = "train"
    pool.iloc[val_idx, pool.columns.get_loc("split")] = "val"
    print("ISIC split CREATED (saved with the manifest in Cell 7)")

assert pool["split"].isin(["train", "val"]).all(), "an image has no split assigned"
overlap = set(pool.loc[pool.split == "train", "group_id"]) & set(pool.loc[pool.split == "val", "group_id"])
assert not overlap, f"group leakage between train and val: {len(overlap)} groups"

# The ISIC split gets the same protection the PAD split has in notebook 02: the numbers are
# asserted, not printed. Cell 7 rewrites the manifest on every run, so without this the only
# record of the previous split is destroyed in the same pass that silently replaces it.
n_train, n_val = int((pool.split == "train").sum()), int((pool.split == "val").sum())
assert (n_train, n_val) == (5364, 1040), (
    f"ISIC split is {n_train:,} train / {n_val:,} val, expected 5,364 / 1,040. "
    + ("The split was LOADED from the manifest, so the manifest itself changed."
       if loaded else
       "The split was just CREATED on this machine - either the pool changed or the library "
       "splits differently. Decide deliberately before Cell 7 overwrites the manifest."))
print(f"split sizes asserted: {n_train:,} train / {n_val:,} val")

print(pool.groupby("split").agg(images=("isic_id", "size"),
                                malignant=("label", "sum"),
                                groups=("group_id", "nunique")).to_string())
print(f"\nmalignant share - train {100*pool.loc[pool.split=='train','label'].mean():.1f}% "
      f"| val {100*pool.loc[pool.split=='val','label'].mean():.1f}%")
print("\ncontributor mix per split (% of each contributor's images):")
print(pd.crosstab(pool["attribution"], pool["split"], normalize="index").mul(100).round(1).to_string())

ISIC split LOADED from: /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/processed/isic_image_manifest.csv
split sizes asserted: 5,364 train / 1,040 val
       images  malignant  groups
split                           
train    5364       3085    4365
val      1040        602     872

malignant share - train 57.5% | val 57.9%

contributor mix per split (% of each contributor's images):
split                                   train   val
attribution                                        
Anonymous                                84.6  15.4
Hospital Italiano de Buenos Aires       100.0   0.0
MILK study team                          83.3  16.7
Memorial Sloan Kettering Cancer Center   85.5  14.5


## 2 · PAD — frozen target-domain split

Notebook 02 created the PAD split at the **patient level** and saved it before subsequent
model-development decisions.

PAD is not one monolithic test set: it contains frozen train, validation and held-out test patients.
Train/validation patients may be used for symptom modelling, fusion, calibration and other development
steps defined in the charter. Test patients are reserved for final evaluation.

Notebook 02 also showed why patient-level splitting matters: **19.1% of patient-lesions have more than
one image**, and a naive image-level split puts another image of the same lesion in train for
**29.3% of test images**.

This notebook reads the frozen split and never re-creates it.


In [ ]:
# Cell 5 · Load the frozen PAD split from notebook 02

pad_split = pd.read_csv(os.path.join(PROCESSED, "pad_split.csv"))

# Make this cell safe to re-run: remove columns left by a previous execution
cols_to_remove = [c for c in ["split", "split_x", "split_y", "lesion_key", "lesion_key_x", "lesion_key_y"]
                  if c in pad.columns]
if cols_to_remove:
    pad = pad.drop(columns=cols_to_remove)

merge_cols = ["img_id", "split"] + (["lesion_key"] if "lesion_key" in pad_split.columns else [])
pad = pad.merge(pad_split[merge_cols], on="img_id", how="left", validate="one_to_one")

assert pad["split"].notna().all(), "Some PAD images have no frozen split. Re-run notebook 02 first."
assert (pad["split"] == "test").sum() == 501, "the frozen PAD test split is no longer 501 images"
assert pad.loc[pad["split"] == "test", "patient_id"].nunique() == 275, "the frozen PAD test split is no longer 275 patients"

if "lesion_key" not in pad.columns:
    pad["lesion_key"] = pad["patient_id"].astype(str) + "::" + pad["lesion_id"].astype(str)

# Project target: malignant = MEL + BCC + SCC; ACK counts as 0
MAP = {"MEL": "melanoma", "BCC": "bcc", "SCC": "scc", "NEV": "nevus", "SEK": "sk", "ACK": "ak"}
pad["cls"] = pad["diagnostic"].map(MAP)
assert pad["cls"].notna().all(), \
    f"unmapped diagnostic codes: {sorted(pad.loc[pad['cls'].isna(), 'diagnostic'].unique())}"

pad["label"] = pad["cls"].isin(["melanoma", "bcc", "scc"]).astype(int)
pad["is_melanoma"] = (pad["cls"] == "melanoma").astype(int)
pad["patient_id"] = pad["patient_id"].astype(str)

print("Frozen PAD split:")
print(pad.groupby("split").agg(images=("img_id", "size"),
                               patients=("patient_id", "nunique"),
                               lesions=("lesion_key", "nunique")).to_string())

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = set(pad.loc[pad["split"] == a, "patient_id"]) & set(pad.loc[pad["split"] == b, "patient_id"])
    assert not overlap, f"Patient overlap between {a} and {b}: {len(overlap)}"
print("\nPatient overlap between every pair of PAD splits: 0 ✓")

pad_dev = pad[pad["split"].isin(["train", "val"])].copy()
print("\nDevelopment target composition:")
print(pad_dev.groupby("split").agg(malignant_images=("label", "sum"),
                                   melanoma_images=("is_melanoma", "sum")).to_string())
print("\nHeld-out test labels are not summarised during development.")

Frozen PAD split:
       images  patients  lesions
split                           
test      501       275      402
train    1440       878     1199
val       357       220      290

Patient overlap between every pair of PAD splits: 0 ✓

Development target composition:
       malignant_images  melanoma_images
split                                   
train               675               35
val                 176                8

Held-out test labels are not summarised during development.


In [ ]:
# Cell 5b · Write the PAD target-domain image manifest

pad_man = pad[
    [
        "img_id",
        "patient_id",
        "lesion_id",
        "lesion_key",
        "split",
        "label",
        "is_melanoma"
    ]
].copy()

pad_man["source"] = "PAD-UFES-20"

pad_manifest_path = os.path.join(
    PROCESSED,
    "pad_image_manifest.csv"
)

pad_man.to_csv(
    pad_manifest_path,
    index=False
)

print(
    pad_man.groupby("split")
           .agg(
               images=("img_id", "size"),
               patients=("patient_id", "nunique"),
               lesions=("lesion_key", "nunique")
           )
           .to_string()
)

print(
    f"\nsaved {len(pad_man):,} rows -> "
    f"{pad_manifest_path}"
)

       images  patients  lesions
split                           
test      501       275      402
train    1440       878     1199
val       357       220      290

saved 2,298 rows -> /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/processed/pad_image_manifest.csv


The official Mendeley metadata remains the source of truth for labels and splits. The **photographs**
do not come from Mendeley: anonymous bulk download is blocked from Colab, so they are fetched from a
public PAD-UFES-20 mirror (`SalmaneExploring/pad-ufes-20` on Hugging Face).

Notebook 01 fetches them from the same repository with the same patterns, so whichever notebook runs
first does the work and the other finds the folder already complete. Cell 5c checks that all 2,298
official filenames are present; **Cell 5e checks the content**: every file must be the same photograph
as one image in PAD's own copy inside ISIC (collection 406), with the same malignant flag.

In [ ]:
# Cell 5c · PAD-UFES-20 image files from a public mirror (downloaded once)
# Same repository and patterns as notebook 01: whichever runs first fetches them, the other skips.
# Metadata and frozen splits remain those created from the official PAD dataset.
import shutil
from pathlib import Path

PAD_IMAGES_DIR = os.path.join(IMAGES_DIR, "pad_original")
os.makedirs(PAD_IMAGES_DIR, exist_ok=True)

expected = pad["img_id"].astype(str).tolist()
expected_stems = {Path(x).stem for x in expected}
assert len(expected_stems) == 2298


def stored_stems():
    return {Path(f).stem for f in os.listdir(PAD_IMAGES_DIR)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))}


if expected_stems <= stored_stems():
    print("all 2,298 PAD images already on Drive - mirror download skipped")
else:
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download

    PAD_MIRROR_DIR = "/content/pad_ufes20_mirror"          # temporary
    snapshot_download(repo_id="SalmaneExploring/pad-ufes-20", repo_type="dataset",
                      allow_patterns=["all_images/**"], local_dir=PAD_MIRROR_DIR)

    mirror_by_stem = {}
    for root, _, files in os.walk(PAD_MIRROR_DIR):
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                mirror_by_stem[Path(f).stem] = os.path.join(root, f)

    missing = expected_stems - set(mirror_by_stem)
    assert not missing, f"{len(missing)} PAD images could not be matched by filename"

    copied = 0
    for img_id in expected:
        stem = Path(img_id).stem
        src = mirror_by_stem[stem]
        filename = os.path.basename(img_id) if Path(img_id).suffix else stem + Path(src).suffix
        dst = os.path.join(PAD_IMAGES_DIR, filename)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            copied += 1
    print(f"new files copied to Drive: {copied:,}")

missing_stored = expected_stems - stored_stems()
print(f"PAD images stored: {len(stored_stems()):,} | still missing: {len(missing_stored):,}")
assert not missing_stored
print("PAD filename check passed ✓  (content is verified in Cell 5e)")

all 2,298 PAD images already on Drive - mirror download skipped
PAD images stored: 2,298 | still missing: 0
PAD filename check passed ✓  (content is verified in Cell 5e)


In [ ]:
# Cell 5d · Add persistent image paths to the PAD manifest

from pathlib import Path

stored_files = [
    f
    for f in os.listdir(PAD_IMAGES_DIR)
    if f.lower().endswith(
        (".png", ".jpg", ".jpeg")
    )
]

stored_by_stem = {
    Path(f).stem: f
    for f in stored_files
}

pad_man["file"] = (
    pad_man["img_id"]
    .astype(str)
    .map(
        lambda x:
        "pad_original/"
        + stored_by_stem[Path(x).stem]
    )
)

assert pad_man["file"].notna().all()
assert len(pad_man) == 2298

pad_manifest_path = os.path.join(
    PROCESSED,
    "pad_image_manifest.csv"
)

pad_man.to_csv(
    pad_manifest_path,
    index=False
)

print(
    pad_man[
        [
            "img_id",
            "split",
            "file"
        ]
    ].head().to_string(
        index=False
    )
)

print(
    f"\nPAD manifest updated -> "
    f"{pad_manifest_path}"
)

print(
    f"rows: {len(pad_man):,}"
)

               img_id split                               file
PAT_1516_1765_530.png  test pad_original/PAT_1516_1765_530.png
   PAT_46_881_939.png train    pad_original/PAT_46_881_939.png
PAT_1545_1867_547.png train pad_original/PAT_1545_1867_547.png
PAT_1989_4061_934.png train pad_original/PAT_1989_4061_934.png
 PAT_684_1302_588.png train  pad_original/PAT_684_1302_588.png

PAD manifest updated -> /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/processed/pad_image_manifest.csv
rows: 2,298


In [ ]:
# Cell 5e · Verify the mirror against PAD's own copy inside ISIC (collection 406)
# A filename match proves nothing about content. Every mirror image must be the same photograph
# as one ISIC-406 image (pHash nearest neighbour), and the malignant flag of that ISIC image must
# agree with the PAD label. Integrity audit only: no model is fitted or tuned here.
# Self-contained: reads everything it needs from disk, so it can be run on its own.
import os, sys, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from tqdm import tqdm
from PIL import Image
try:
    import imagehash
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imagehash"])
    import imagehash
!pip install -q isic-cli

# 1 · PAD's official copy inside ISIC, downloaded once to temporary local disk
ISIC_PAD_DIR = "/content/isic_pad_copy"          # temporary, not part of the project
os.makedirs(ISIC_PAD_DIR, exist_ok=True)
if len([f for f in os.listdir(ISIC_PAD_DIR) if f.lower().endswith((".jpg", ".png"))]) < 2298:
    !isic image download --collections 406 "$ISIC_PAD_DIR"

# 2 · The mirror files we actually use, on Google Drive
PAD_IMAGES_DIR = os.path.join(IMAGES_DIR, "pad_original")
isic_files   = sorted(f for f in os.listdir(ISIC_PAD_DIR) if f.lower().endswith((".jpg", ".png")))
mirror_names = sorted(f for f in os.listdir(PAD_IMAGES_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg")))
assert len(isic_files) == 2298, f"expected 2,298 ISIC-406 images, found {len(isic_files):,}"
assert len(mirror_names) == 2298, f"expected 2,298 PAD files on Drive, found {len(mirror_names):,}"
print(f"ISIC collection 406 images: {len(isic_files):,} | mirror images: {len(mirror_names):,}")


# 3 · Perceptual hashes, read in parallel (Drive reads are network-bound, not CPU-bound)
def phash_file(p):
    with Image.open(p) as im:
        return imagehash.phash(im)


def hashes(paths, workers=16):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(tqdm(ex.map(phash_file, paths), total=len(paths)))   # order is preserved


def to_bits(hs):
    return np.unpackbits(
        np.array([np.frombuffer(bytes.fromhex(str(h)), dtype=np.uint8) for h in hs]),
        axis=1).astype(np.float32)


Hi = to_bits(hashes([os.path.join(ISIC_PAD_DIR, f) for f in isic_files]))
Hm = to_bits(hashes([os.path.join(PAD_IMAGES_DIR, f) for f in mirror_names]))

# 4 · Nearest ISIC-406 image for every mirror image
dist = Hm @ (1 - Hi).T + (1 - Hm) @ Hi.T                  # 2,298 x 2,298 Hamming distances
nearest, dmin = dist.argmin(axis=1), dist.min(axis=1)

match = pd.DataFrame({"img_id": mirror_names,
                      "isic_id": [Path(isic_files[j]).stem for j in nearest],
                      "distance": dmin.astype(int)})

# 5 · Labels on both sides, straight from the source files
isic_meta = pd.read_csv(os.path.join(ISIC_PAD_DIR, "metadata.csv"))
isic_malignant = isic_meta.set_index("isic_id")["diagnosis_1"].eq("Malignant").astype(int)

pad_meta = pd.read_csv(os.path.join(RAW, "pad_metadata_raw.csv"))
pad_label = pad_meta.set_index("img_id")["diagnostic"].isin(["MEL", "BCC", "SCC"]).astype(int)

match["isic_malignant"] = match["isic_id"].map(isic_malignant)
match["pad_label"] = match["img_id"].map(pad_label)
assert match[["isic_malignant", "pad_label"]].notna().all().all(), "an image could not be matched to its label"

# 6 · Verdict
print("\nnearest-neighbour distance, mirror -> ISIC 406:")
for k in [0, 2, 4, 6]:
    print(f"  <= {k}: {100 * (match['distance'] <= k).mean():5.1f}%")
print(f"distinct ISIC images matched: {match['isic_id'].nunique():,} of {len(match):,} "
      "(2,298 = one-to-one)")

agree = (match["isic_malignant"] == match["pad_label"]).mean()
print(f"malignant flag agreement (mirror name vs. ISIC copy): {100 * agree:.1f}%")

bad = match[(match["distance"] > 6) | (match["isic_malignant"] != match["pad_label"])]
print(f"\nimages to inspect (far match or label disagreement): {len(bad)}")
if len(bad):
    print(bad.head(20).to_string(index=False))

match.to_csv(os.path.join(PROCESSED, "pad_mirror_verification.csv"), index=False)
print(f"\nsaved -> {os.path.join(PROCESSED, 'pad_mirror_verification.csv')}")

If you have been granted special permissions, logging in with `isic user login` might return more data.


Successfully downloaded 2,298 images to /content/isic_pad_copy/.
Successfully wrote 2,298 metadata records to /content/isic_pad_copy/metadata.csv.
Successfully wrote attributions to /content/isic_pad_copy/attribution.txt.
Successfully wrote 1 license(s) to /content/isic_pad_copy/licenses.
ISIC collection 406 images: 2,298 | mirror images: 2,298


100%|██████████| 2298/2298 [01:33<00:00, 24.53it/s]



nearest-neighbour distance, mirror -> ISIC 406:
  <= 0:  87.6%
  <= 2:  99.9%
  <= 4: 100.0%
  <= 6: 100.0%
distinct ISIC images matched: 2,282 of 2,298 (2,298 = one-to-one)
malignant flag agreement (mirror name vs. ISIC copy): 99.7%

images to inspect (far match or label disagreement): 7
              img_id      isic_id  distance  isic_malignant  pad_label
 PAT_202_307_424.png ISIC_2533328         0               0          1
  PAT_38_1002_34.png ISIC_3928650         0               0          1
 PAT_38_1002_668.png ISIC_1609857         2               0          1
PAT_419_2767_323.png ISIC_3622062         0               1          0
  PAT_691_1311_2.png ISIC_1393731         0               0          1
PAT_691_1311_890.png ISIC_0329449         0               0          1
PAT_759_1538_566.png ISIC_0256903         2               0          1

saved -> /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/processed/pad_mirror_verification.csv


**Mirror verification (Cell 5e).** Every one of the 2,298 mirror files is the same photograph as an image in PAD's official copy inside ISIC (collection 406): 87.6% have an identical perceptual hash and 100% are within distance 4 (the rest differ only by PNG vs. JPEG compression). 2,282 distinct ISIC images are matched; the 16 shared matches are near-identical repeat photographs. The malignant flag agrees for 99.7% of images. The 7 disagreements are all distance 0–2, i.e. the same photograph labelled differently by ISIC, consistently per lesion (two lesions account for 4 of them). They reflect ISIC's re-curation, not a mirror error. **The Mendeley labels remain the source of truth and are not changed.** Six of the seven are PAD-malignant against ISIC-benign and only one is the reverse, so the direction is conservative: the Mendeley labels call slightly more cancer than ISIC's re-curation, across five lesions out of 1,891.

## 3 · Features for the symptom model

Three rules, each one a lesson from something already found in this project:

**One row per lesion, not per photograph.** Notebook 02 verified that the symptom answers, age and body
site are identical across every image of the same `patient_id::lesion_id`, so repeating them once per photo
would silently weight lesions by how many times they happened to be photographed. The honest metadata
baseline (**AUC 0.860**, variant D in notebook 02: all lesions, imputed, no missingness indicator) and the
validation scores at the operating point were measured on this unit; this file has to match it.

**The feature list is explicit.** Anything not named below cannot reach the model, so forgetting to drop a
column is impossible by construction.

**`biopsed` is banned by name.** It records whether a biopsy was performed — which happens *because* the
lesion was suspicious. It is the outcome wearing a different hat. So are `diagnostic`, `cls` and `label`.
The assertion makes the ban enforceable rather than aspirational.

In [ ]:
# Cell 6 · Feature matrix, one row per lesion, with an enforced ban list
FEATURES_NUM = ["age"]
FEATURES_CAT = ["region", "itch", "grew", "hurt", "changed", "bleed", "elevation"]
FEATURES = FEATURES_NUM + FEATURES_CAT

FORBIDDEN = ["diagnostic", "cls", "label", "is_melanoma", "biopsed", "gender",
             "smoke", "drink", "pesticide", "skin_cancer_history", "cancer_history",
             "has_piped_water", "has_sewage_system", "fitspatrick",
             "diameter_1", "diameter_2", "background_father", "background_mother"]

leaks = [c for c in FEATURES if c in FORBIDDEN]
assert not leaks, f"a banned column reached the feature list: {leaks}"

# Collapse to one row per lesion, and verify that collapsing loses nothing
varies = [c for c in FEATURES + ["label"]
          if (pad.groupby("lesion_key")[c].nunique(dropna=False) > 1).any()]
assert not varies, f"these fields differ between images of the same lesion: {varies}"

lesions = pad.sort_values("img_id").drop_duplicates("lesion_key").copy()
feat = lesions[["lesion_key", "patient_id", "split", "label"] + FEATURES].copy()
feat.to_csv(os.path.join(PROCESSED, "pad_symptom_features.csv"), index=False)

print(f"images {len(pad):,} -> lesions {len(feat):,}")
print(feat.groupby("split").agg(lesions=("lesion_key", "size"), malignant=("label", "sum")).to_string())
print("\nfeatures used:", FEATURES)
print("\nexcluded on purpose:")
print("  biopsed        - records that a biopsy happened, i.e. the outcome")
print("  fitspatrick    - reporting only; recorded for 65% of images (see notebook 02)")
print("  gender/smoke/drink/history/diameter - optional fields, missing mainly for benign cases")

dev_feat = feat[feat.split.isin(["train", "val"])]
print("\nmissing values per feature, development lesions")
print("(kept as NaN; the models impute inside the pipeline with no missingness indicator - notebook 02, variant D):")
print(dev_feat[FEATURES].isna().sum().to_string())
print(f"\nsaved -> {os.path.join(PROCESSED, 'pad_symptom_features.csv')}")

images 2,298 -> lesions 1,891
       lesions  malignant
split                    
test       402        178
train     1199        528
val        290        127

features used: ['age', 'region', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation']

excluded on purpose:
  biopsed        - records that a biopsy happened, i.e. the outcome
  fitspatrick    - reporting only; recorded for 65% of images (see notebook 02)
  gender/smoke/drink/history/diameter - optional fields, missing mainly for benign cases

missing values per feature, development lesions
(kept as NaN; the models impute inside the pipeline with no missingness indicator - notebook 02, variant D):
age            0
region         0
itch           5
grew         302
hurt           5
changed      298
bleed          5
elevation      2

saved -> /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/processed/pad_symptom_features.csv


## 4 · The image manifest — this is the hand-off to the vision model

One row per image: where the file is, what the label is, which split it belongs to, and which group it
cannot be separated from. The vision notebook reads **only this file** and never re-derives a split.


In [ ]:
# Cell 7 · Write the manifest the vision model consumes
# The body-site column depends on the ISIC taxonomy version: take whichever is present.
site_col = next((c for c in ["anatom_site_1", "anatom_site_general", "anatom_site_special"]
                 if c in pool.columns), None)
extra = [c for c in ["attribution", "age_approx", "sex", site_col, "fitzpatrick_skin_type"]
         if c and c in pool.columns]

# Reporting-only columns: for subgroup analysis, NEVER model inputs. attribution and
# fitzpatrick_skin_type are contributor proxies (MSK = 99.4% benign, notebook 01).
REPORTING_ONLY = extra
print("reporting-only columns (never model inputs):", REPORTING_ONLY)

man = pool[["isic_id", "label", "label_family", "split", "group_id"] + extra].copy()
man["is_melanoma"] = pool["is_melanoma"].astype(int)
man["source"] = "ISIC"
man["file"] = "isic_512/" + man["isic_id"] + ".jpg"     # path inside images_isic_512.tar

man.to_csv(ISIC_MANIFEST, index=False)

print(man.groupby(["split", "label"]).size().unstack(fill_value=0).to_string())
print(f"\nsaved {len(man):,} rows -> {ISIC_MANIFEST}")

pos = int(man.loc[man.split == "train", "label"].sum())
neg = int((man.loc[man.split == "train", "label"] == 0).sum())
print(f"\nclass weight suggestion for training: pos_weight = {neg / max(pos, 1):.2f}  ({neg:,} neg / {pos:,} pos)")

reporting-only columns (never model inputs): ['attribution', 'age_approx', 'sex', 'anatom_site_1', 'fitzpatrick_skin_type']
label     0     1
split            
train  2279  3085
val     438   602

saved 6,404 rows -> /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/processed/isic_image_manifest.csv

class weight suggestion for training: pos_weight = 0.74  (2,279 neg / 3,085 pos)


## 5 · Frozen prediction aggregation

The model produces scores per image, while final triage is evaluated per patient.

The aggregation rule is frozen before test evaluation:

1. **Image → lesion:** use the maximum risk score across images of the same lesion.
2. **Lesion → patient:** use the maximum fused risk score across lesions belonging to the same patient.

The maximum is used because triage priority should reflect the patient's most suspicious submitted lesion,
rather than dilute it by averaging with lower-risk images or lesions.

In [ ]:
# Cell 7b · Save the frozen aggregation policy

import json

aggregation_policy = {
    "image_to_lesion": "max",
    "lesion_to_patient": "max",
    "rationale": (
        "Patient triage priority is determined by the most "
        "suspicious submitted image/lesion; averaging could "
        "dilute a high-risk lesion."
    )
}

aggregation_path = os.path.join(
    PROCESSED,
    "aggregation_policy.json"
)

with open(
    aggregation_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        aggregation_policy,
        f,
        indent=2
    )

print(
    "Frozen aggregation policy:"
)

print(
    json.dumps(
        aggregation_policy,
        indent=2
    )
)

print(
    f"\nsaved -> {aggregation_path}"
)

Frozen aggregation policy:
{
  "image_to_lesion": "max",
  "lesion_to_patient": "max",
  "rationale": "Patient triage priority is determined by the most suspicious submitted image/lesion; averaging could dilute a high-risk lesion."
}

saved -> /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/processed/aggregation_policy.json


In [ ]:
# Cell 8 · Final integrity checks - these must all pass before modelling starts
checks = []

# ISIC
ov = set(pool.loc[pool.split == "train", "group_id"]) & set(pool.loc[pool.split == "val", "group_id"])
checks.append(("ISIC: no group spans train and val", len(ov) == 0, f"{len(ov)} groups"))
checks.append(("ISIC: 6,404 images after all exclusions", len(pool) == 6404, f"{len(pool):,}"))
checks.append(("ISIC: only MEL/BCC/SCC/BENIGN families",
               pool["label_family"].isin(["MEL", "BCC", "SCC", "BENIGN"]).all(), ""))
checks.append(("ISIC: no duplicate photograph left", not pool["isic_id"].isin(dup_ids).any(), ""))
checks.append(("ISIC: manifest ids equal the pool", set(man["isic_id"]) == set(pool["isic_id"]), ""))
pad_in_isic = pool["attribution"].str.contains("UFES|Esp.rito Santo", na=False, regex=True).sum()
checks.append(("ISIC: no PAD image inside the training pool", pad_in_isic == 0, f"{pad_in_isic} images"))
checks.append(("ISIC: manifest has no missing labels", man["label"].notna().all(), ""))

# Only contributors large enough to be split meaningfully: a 6-image contributor with a handful of
# patients cannot be balanced under patient-level grouping, and cannot bias validation either.
MIN_IMAGES = 50
counts = pool["attribution"].value_counts()
large = counts[counts >= MIN_IMAGES].index
share = pd.crosstab(pool["attribution"], pool["split"], normalize="index").loc[large]
worst = float(share.max(axis=1).max()) if not share.empty else 0.0
checks.append((f"ISIC: no contributor >= {MIN_IMAGES} images in one split", worst < 0.95,
               f"max {100 * worst:.0f}% ({', '.join(s[:12] for s in large)})"))

# PAD
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    o = set(pad.loc[pad.split == a, "patient_id"]) & set(pad.loc[pad.split == b, "patient_id"])
    checks.append((f"PAD: no patient spans {a} and {b}", len(o) == 0, f"{len(o)} patients"))
    o = set(pad.loc[pad.split == a, "lesion_key"]) & set(pad.loc[pad.split == b, "lesion_key"])
    checks.append((f"PAD: no lesion spans {a} and {b}", len(o) == 0, f"{len(o)} lesions"))

checks.append(("PAD: test is 501 images / 275 patients",
               (pad.split == "test").sum() == 501
               and pad.loc[pad.split == "test", "patient_id"].nunique() == 275, ""))

# Symptom features
ID_COLS = {"lesion_key", "patient_id", "split", "label"}
offenders = (set(feat.columns) - ID_COLS) & set(FORBIDDEN)
checks.append(("Features: no banned column", not offenders, str(offenders)))
checks.append(("Features: one row per lesion", feat["lesion_key"].is_unique, ""))
checks.append(("Features: 1,489 dev + 402 test lesions",
               feat.split.isin(["train", "val"]).sum() == 1489 and (feat.split == "test").sum() == 402, ""))

print(f"{'check':<52} {'result':<8} detail")
print("-" * 78)
for name, ok, detail in checks:
    print(f"{name:<52} {'PASS' if ok else 'FAIL':<8} {detail}")

assert all(ok for _, ok, _ in checks), "an integrity check failed - fix it before training anything"
print("\nAll checks passed. The splits are frozen.")

check                                                result   detail
------------------------------------------------------------------------------
ISIC: no group spans train and val                   PASS     0 groups
ISIC: 6,404 images after all exclusions              PASS     6,404
ISIC: only MEL/BCC/SCC/BENIGN families               PASS     
ISIC: no duplicate photograph left                   PASS     
ISIC: manifest ids equal the pool                    PASS     
ISIC: no PAD image inside the training pool          PASS     0 images
ISIC: manifest has no missing labels                 PASS     
ISIC: no contributor >= 50 images in one split       PASS     max 85% (MILK study t, Memorial Slo, Anonymous)
PAD: no patient spans train and val                  PASS     0 patients
PAD: no lesion spans train and val                   PASS     0 lesions
PAD: no patient spans train and test                 PASS     0 patients
PAD: no lesion spans train and test                  PASS     

In [ ]:
# Cell 9 · Generate data/data_dictionary.md (a required deliverable)
DD = os.path.join(PROJECT, "data", "data_dictionary.md")

n_pad_excluded = int(isic["exclusion_reason"].eq("pad_test_set").sum())
n_benchmark    = int(isic["exclusion_reason"].eq("unlabelled_benchmark").sum())
site_note      = f", `{site_col}`" if site_col else ""

lines = [
    "# Data dictionary",
    "",
    "Generated by `notebooks/03_preprocessing.ipynb`. Do not edit by hand.",
    "",
    "## Sources",
    "",
    "| Source | Role | Rows | Licence |",
    "|---|---|---|---|",
    f"| ISIC Archive, clinical close-ups | labelled image-model pool | {len(pool):,} images | CC-0 / CC-BY / CC-BY-NC, per image |",
    f"| PAD-UFES-20 | target domain, frozen patient-level train/val/test split | {len(pad):,} images | CC BY 4.0 |",
    "",
    "Labels and splits for PAD come from the official Mendeley metadata. The image files come from a public",
    "mirror of the same dataset; all 2,298 filenames match the official ids, and every file was checked against",
    "PAD's own copy inside ISIC (collection 406) by perceptual hash (notebook 03, Cell 5e):",
    "",
    "- 100% of mirror files are within pHash distance 4 of an ISIC-406 image, and 87.6% are identical; the rest",
    "  differ only by PNG vs. JPEG compression.",
    "- 2,282 distinct ISIC images are matched; the 16 shared matches are near-identical repeat photographs.",
    "- The malignant flag agrees for 99.7% of images. The 7 disagreements are the same photographs (distance 0–2)",
    "  labelled differently by ISIC, consistently per lesion. They reflect ISIC's re-curation, not a mirror error;",
    "  **the Mendeley labels are kept**.",
    "",
    "## Exclusions applied to the ISIC pool",
    "",
    "| Rule | Images removed | Decided in |",
    "|---|---|---|",
    f"| PAD-UFES-20 copy inside ISIC (held out, managed separately) | {n_pad_excluded:,} | notebook 01, Cell 4 |",
    f"| Unlabelled MILK10k benchmark | {n_benchmark:,} | notebook 01, Cell 4 |",
    f"| Indeterminate diagnosis (123 actinic keratoses, 1 atypical melanocytic neoplasm) | {int(drop_indet.sum()):,} | notebook 01, Cell 5 |",
    f"| Collision lesions, malignant component unidentified | {int(drop_other.sum()):,} | notebook 01, Cell 5b |",
    f"| Duplicate photographs (same MILK photo under two lesion ids) | {int(drop_dup.sum()):,} | notebook 01, Cell 16d |",
    f"| **Images used for the image model** | **{len(pool):,}** ({int(pool['label'].sum()):,} malignant, "
    f"{100 * pool['label'].mean():.1f}%; {int(pool['is_melanoma'].sum()):,} melanomas) | |",
    "",
    "## `isic_image_manifest.csv`",
    "",
    "| Column | Meaning |",
    "|---|---|",
    "| `isic_id` | ISIC image identifier |",
    "| `file` | path inside `data/raw/images/images_isic_512.tar` |",
    "| `label` | 1 = MEL, BCC or SCC (frozen label policy), 0 = not malignant |",
    "| `label_family` | MEL / BCC / SCC / BENIGN; SCC includes keratoacanthoma |",
    "| `is_melanoma` | 1 for melanoma; reported separately |",
    f"| `split` | `train` ({int((pool.split == 'train').sum()):,}) or `val` ({int((pool.split == 'val').sum()):,}); "
    "created once and frozen; no patient and no lesion spans both. Stratified by contributor and label. "
    "Hospital Italiano (6 images, a handful of patients) sits entirely in one split: too small to divide "
    "under patient-level grouping |",
    "| `group_id` | `<contributor>::P::<patient>`, else `::L::<lesion>`, else `::I::<image>`. "
    "Patient comes first because 1,247 images come from only 79 patients (MSK 1,241, Hospital Italiano 6), "
    "while barely 0.2% of lesions have more than one image |",
    f"| `attribution`, `age_approx`, `sex`{site_note}, `fitzpatrick_skin_type` | **reporting only, never model inputs**: "
    "the contributor alone predicts malignancy with AUC 0.729, and Fitzpatrick is recorded almost only for MSK |",
    "| `source` | always `ISIC` |",
    "",
    "## `pad_image_manifest.csv`",
    "",
    "| Column | Meaning |",
    "|---|---|",
    "| `img_id` | PAD filename, as published |",
    "| `file` | path relative to `data/raw/images/`, e.g. `pad_original/PAT_46_881_939.png` |",
    "| `patient_id`, `lesion_id` | as published |",
    "| `lesion_key` | `patient_id::lesion_id`. PAD reuses 250 lesion ids across patients, so the raw id is not a lesion |",
    "| `label` | 1 = MEL, BCC or SCC. ACK (pre-cancerous) counts as 0 |",
    "| `is_melanoma` | 1 for MEL |",
    "| `split` | patient-level, frozen in notebook 02 (train 1,440 · val 357 · test 501 images); "
    "`test` is untouchable until final evaluation |",
    "| `source` | always `PAD-UFES-20` |",
    "",
    "## `pad_symptom_features.csv`",
    "",
    "One row per lesion (`patient_id::lesion_id`), not per photograph: the answers are identical across every",
    "image of a lesion, so one row per photo would weight lesions by how often they were photographed.",
    "",
    "| Column | Meaning |",
    "|---|---|",
    "| `lesion_key` | `patient_id::lesion_id`; the row identifier |",
    "| `patient_id` | for patient-level aggregation (max over lesions) |",
    "| `age` | years, as reported |",
    "| `region` | body site, PAD vocabulary |",
    "| `itch`, `grew`, `hurt`, `changed`, `bleed`, `elevation` | the six questions the app asks; True / False / missing |",
    "| `label` | 1 = MEL, BCC or SCC. ACK counts as 0 |",
    "| `split` | patient-level; `test` is frozen and untouchable |",
    "",
    "**Fields deliberately excluded from the features:** `biopsed` (records that a biopsy took place, i.e. the "
    "outcome), and the optional fields `gender`, `smoke`, `drink`, `pesticide`, `skin_cancer_history`, "
    "`cancer_history`, `has_piped_water`, `has_sewage_system`, `fitspatrick`, `diameter_1`, `diameter_2`. "
    "Those are missing mainly for benign cases, so their *absence* predicts the label for administrative "
    "rather than medical reasons (missing optional-field count alone: AUC 0.839).",
    "",
    "**Missing values are kept as `NaN` in the file.** `grew` and `changed` are unanswered for 20% of development "
    "lesions, 87–88% of them malignant, so a missingness indicator would leak the label. The models therefore "
    "impute inside the training pipeline (most frequent answer) and never receive an indicator. Imputing the "
    "usual answer, \"no\", makes these mostly malignant lesions look *less* suspicious, so the policy can only "
    "lower the baseline, never inflate it. The honest baseline is **AUC 0.860** (notebook 02, variant D).",
    "",
    "**Who filled the form:** senior medical students during the visit (Pacheco et al., 2020). App users will "
    "answer alone; flipping 10% of answers lowers the baseline's sensitivity at the primary operating point "
    "from 0.52 to 0.47 (notebook 02, Cell 8c).",
    "",
    "## `aggregation_policy.json`",
    "",
    "The frozen rule for turning image scores into the patient score that triage is evaluated on:",
    "`image -> lesion` by **max**, `lesion -> patient` by **max**. A patient's priority is their most suspicious "
    "lesion; averaging would dilute it. A patient who submitted several photographs gets several chances at a "
    "high score, which raises both sensitivity and false alarms for that patient.",
    "",
    "## Baselines the models are measured against (notebook 02)",
    "",
    "- `baseline_metadata_val.csv`: questionnaire-only sensitivity, PPV and flag rate on validation patients, "
    "9 scenarios (5/10/15 slots for 100 arrivals × 5/10/20% prevalence). Primary scenario (10 slots, 10% "
    "prevalence): sensitivity 0.52, PPV 0.53.",
    "- `val_scores_metadata.csv`: per-patient questionnaire scores on the same 220 validation patients, for a "
    "paired comparison with the image and fusion models.",
    "",
    "## Transformations",
    "",
    f"- ISIC images resized so the long side is {IMG_SIZE} px, saved as JPEG quality 92, packed into one tar.",
    "- PAD images are stored at their original size under `data/raw/images/pad_original/`.",
    f"- All splits use `random_state={SEED}`, are created once and loaded on every later run.",
    "- PAD missing values arrive as `UNK` or an empty string, and are normalised to `NaN` on load.",
    "",
]

open(DD, "w", encoding="utf-8").write("\n".join(lines))
print(f"written -> {DD}  ({len(lines)} lines)")
print("\n".join(lines[:34]))

written -> /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/data/data_dictionary.md  (98 lines)
# Data dictionary

Generated by `notebooks/03_preprocessing.ipynb`. Do not edit by hand.

## Sources

| Source | Role | Rows | Licence |
|---|---|---|---|
| ISIC Archive, clinical close-ups | labelled image-model pool | 6,404 images | CC-0 / CC-BY / CC-BY-NC, per image |
| PAD-UFES-20 | target domain, frozen patient-level train/val/test split | 2,298 images | CC BY 4.0 |

Labels and splits for PAD come from the official Mendeley metadata. The image files come from a public
mirror of the same dataset; all 2,298 filenames match the official ids, and every file was checked against
PAD's own copy inside ISIC (collection 406) by perceptual hash (notebook 03, Cell 5e):

- 100% of mirror files are within pHash distance 4 of an ISIC-406 image, and 87.6% are identical; the rest
  differ only by PNG vs. JPEG compression.
- 2,282 distinct ISIC images are matched; 

## Hand-off to modelling

The files written above are the source of truth for every downstream experiment.

| File | What it is |
|---|---|
| `data/processed/isic_image_manifest.csv` + `data/raw/images/images_isic_512.tar` | the frozen ISIC pool and split, 6,404 images. The image model may consume it directly or rebuild an equivalent index from the same sources and the same rules — the CV branch does the latter and reproduces every count |
| `data/processed/pad_image_manifest.csv` + `data/raw/images/pad_original/` | the frozen PAD patient split, 2,298 images. Fine-tuning uses train patients, validation is for model selection, test is opened once |
| `data/processed/pad_symptom_features.csv` | lesion-level structured features for the symptom model |
| `data/processed/aggregation_policy.json` | image → lesion → patient scoring: max, then max |
| `data/processed/baseline_metadata_val.csv`, `val_scores_metadata.csv` (notebook 02) | **the questionnaire baseline**: logistic regression, variant D, patient AUC 0.867. Specified before any model was fitted and never tuned on validation, which is why it is the bar |
| `data/processed/symptom_scores_fusion.csv` (notebook 04) | calibrated lesion-level symptom scores for a fitted fusion layer. Notebook 04 also reaches a higher patient AUC (0.890), but that model was **selected on the same validation patients**, so it is reported as a study and not used as the bar |

**Rules the modelling notebooks inherit:**
1. **Model selection, early stopping, calibration and thresholds use PAD validation patients**, not the ISIC validation split. Inside ISIC the contributor alone reaches AUC 0.729, so an ISIC-internal score can measure the institution rather than the disease.
2. **Augment colour** (saturation, hue), not only geometry: MSK images carry half the saturation of MILK even at 512 px, a contributor fingerprint the model must not learn.
3. **Augment blur**: PAD phone photos are **2.6× less sharp than MILK and 4.4× less sharp than MSK**, measured at the training resolution (notebook 01, section 12). A quarter of them are below 512 px on the long side and are never upscaled.
4. **Reporting-only columns** (`attribution`, `fitzpatrick_skin_type`, age, sex, site in the ISIC manifest) are never model inputs.
5. **Compare against the questionnaire baseline on the same 220 validation patients** — notebook 02's logistic regression (patient AUC 0.867), not notebook 04's selected model. A bar chosen on the evaluation set is not a bar. ΔAUC and Δsensitivity at the primary operating point (10 slots, 10% prevalence), with paired-bootstrap CIs, plus melanoma sensitivity reported separately.
6. **PAD test patients are opened once**, for the final evaluation.

**Next:** `04_symptom_model.ipynb` — can the questionnaire baseline be improved, what does the model learn, and is it calibrated. Then the image model, and the fusion layer.